# 건설현장 RAG 검색 성능 평가 — 정리본

원본 `Capstone_LLM.ipynb`(124개 셀)에서 **검색 성능 평가만 분리**했습니다. Qwen3-VL 이미지 추론·보고서 생성 실험, 임시 출력, 수동 디버깅 셀은 제거했습니다.

- **검색 대상**: 위험성평가 판단 기준(`rule`)과 사례(`case`)
- **검색 방법**: BGE-M3, BM25, Hybrid(RRF)
- **선택 실험**: 가중치 RRF, BGE reranker, RRF+reranker Fusion
- **지표**: Recall@5, Precision@5, MRR@5
- **평가 데이터**: 노트북의 25개 질문과 후반부 수동 검토·중복 ID 정정을 반영한 `rag_test_queries_reviewed.csv`

### 프로젝트 파일 배치
같은 디렉터리에 `rag_retrieval_evaluation.py`, `rag_test_queries_reviewed.csv`를 배치하세요. 네트워크 접속이 가능한 환경에서는 첫 실행 시 임베딩 모델 가중치를 내려받습니다.

**주의:** 원본 위험 데이터 CSV는 첨부되지 않아 검색 성능 수치는 여기서 재현하지 않았습니다. 원본과 동일 데이터·환경에서 확인하세요.

In [ ]:
from pathlib import Path
from rag_retrieval_evaluation import load_documents, load_test_queries

DATA_PATH = Path("../../risk_data_clean.csv")
QUERY_PATH = Path("rag_test_queries_reviewed.csv")
OUTPUT_DIR = Path("evaluation/rag_results")

assert DATA_PATH.exists(), f"데이터셋이 없습니다: {DATA_PATH}"
assert QUERY_PATH.exists(), f"평가 질문이 없습니다: {QUERY_PATH}"

documents, id_map = load_documents(DATA_PATH)
tests = load_test_queries(QUERY_PATH, id_map, {d["id"] for d in documents})
print(f"검색 문서: {len(documents)}개 / 평가 질문: {len(tests)}개")

## 기본 검색 성능 비교
BGE-M3, BM25, Hybrid(RRF)를 **동일한 질문과 정답 문서**로 평가합니다. 모델 다운로드 및 임베딩 생성에 시간이 걸릴 수 있습니다.

In [ ]:
from rag_retrieval_evaluation import Retriever, evaluate

retriever = Retriever(documents)  # CPU/GPU 자동 선택
detail_df, summary_df = evaluate(retriever, tests, top_k=5, candidate_k=10)
display(summary_df.round(4))

## 선택 실험: 가중치 RRF / Reranker / Fusion
가중치 RRF는 추가 모델 다운로드가 없지만, Reranker는 `BAAI/bge-reranker-v2-m3` 모델을 추가로 불러옵니다. 필요할 때만 실행하세요.

In [ ]:
RUN_WEIGHT_SWEEP = False
RUN_RERANKER = False

from rag_retrieval_evaluation import Reranker
reranker = Reranker() if RUN_RERANKER else None

if RUN_WEIGHT_SWEEP or RUN_RERANKER:
    detail_df, summary_df = evaluate(
        retriever, tests, top_k=5, candidate_k=10,
        weight_sweep=RUN_WEIGHT_SWEEP, reranker=reranker,
    )
    display(summary_df.round(4))
else:
    print("선택 실험을 건너뜁니다. 필요하면 위의 플래그를 True로 변경하세요.")

## 결과 CSV 저장
기본 검색 실험은 항상 저장되며, 선택 실험을 실행한 경우 그 결과가 함께 저장됩니다.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
detail_path = OUTPUT_DIR / "rag_evaluation_detail.csv"
summary_path = OUTPUT_DIR / "rag_evaluation_summary.csv"
detail_df.to_csv(detail_path, index=False, encoding="utf-8-sig")
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
print(f"상세 결과: {detail_path}")
print(f"요약 결과: {summary_path}")